# Line-of-sight (LOS) visibility analysis
This notebook was developed as part of the Master’s thesis of June Rossen and is one of five notebooks used to prepare data for a wind power placement multi-objective optimization.

The goal of this notebook is to analyse the potential visibility between prospective turbine locations and areas containing inhabitants. Its result is needed to run the second notebook, 1_2_1_DataPrep_adding_evaluation_criteria.ipynb .

## Notebook workflow
- Start by defining key variables and downloading necessary data
- Calculate the horizon cutoff distance with a 200m turbines and a 1.7m observer
- Find all of the potential turbine/inhabited cell pairs within this horizon cutoff distance from each other (or another radius of choice, i.e. 20 km)
- Do a coarse filter on those pairs, sampling every 1000m the DEM in vector format, to remove pairs where the line of sight is blocked even at the coarse resolution
- Do a fine filter on the resulting pairs, following the same principle but sampling a DEM every 100m, to have a finer view of the potential line of sights
- Create a dataframe from the resulting pairs, and group the inhabitant count by potential wind turbine location, in order to have an estimate of number of inhabitants that would see each potential wind turbine
- Save the resulting dataframe to use in the 1_2_1_DataPrep_adding_evaluation_criteria.ipynb file

In [1]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
from shapely.geometry import Point, LineString
from scipy.spatial import cKDTree
from joblib import Parallel, delayed
from tqdm import tqdm
import math
import pandas as pd

In [2]:
DEM_PATH = "data/DEM_100m.tif"
turb_gdf = gpd.read_file("data/SSB_tech_legal_filtered.gpkg")
pop_gdf = gpd.read_file("data/SSB_populated.gpkg")
dem_gdf = gpd.read_file("data/SSB_land.gpkg")


TURBINE_HEIGHT = 200.0  # m above ground
OBSERVER_HEIGHT = 1.7   # population eye-level
EARTH_RADIUS = 6371000.0  # m

In [3]:
turb_gdf = turb_gdf[["SSBid", "mean_elevation", "geometry"]]

pop_gdf = pop_gdf.merge(
    dem_gdf[["SSBid", "mean_elevation"]],
    left_on="SSBid",        # Column in the gdf
    right_on="SSBid",       # Column in the df
    how="left"              # Use 'left' to keep all rows from the gdf
)

dem_gdf = dem_gdf[["SSBid", "mean_elevation", "geometry"]]

# Extract centroid coordinates
turb_coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in turb_gdf.geometry])
pop_coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in pop_gdf.geometry])

In [4]:
#display(turb_gdf, turb_gdf.info(), turb_gdf.crs)
#display(pop_gdf, pop_gdf.info(), pop_gdf.crs)
#display(dem_gdf, dem_gdf.info(), dem_gdf.crs)

In [5]:
# Calculate the physical horizon distance (turbine 200 m high, observer 1.7 m) in order to reduce the number of turbine-population pairs that will have to be evaluated,
# uses the Pythagorian theorem twice (simplified considering turbine and observer height small relative to Earth's radius) and adds the results to obtain d_max
d_max = math.sqrt(2 * EARTH_RADIUS * TURBINE_HEIGHT) + math.sqrt(2 * EARTH_RADIUS * OBSERVER_HEIGHT)
print(f"Using horizon cutoff ≈ {d_max/1000:.1f} km")

# KD-tree allows for a faster look-up using query_ball_point
turb_tree = cKDTree(turb_coords)

# For each population cell, find turbines within d_max, decided to be 20 km instead of the horizon distance, to reduce the number of possible pairs
print("Querying turbine neighbors within horizon distance...")
neighbor_lists = turb_tree.query_ball_point(pop_coords, r=20000)

pairs = [(turb_i, pop_i)
         for pop_i, turb_list in enumerate(neighbor_lists)
         for turb_i in turb_list]

print(f"Candidate pairs after horizon filter: {len(pairs):,}")


Using horizon cutoff ≈ 55.1 km
Querying turbine neighbors within horizon distance...
Candidate pairs after horizon filter: 21,078,640


In [9]:
with rasterio.open(DEM_PATH) as src:
    dem = src.read(1, masked=True)
    transform = src.transform

def sample_dem(points):
    cols, rows = ~transform * (points[:,0], points[:,1])
    rows, cols = np.floor(rows).astype(int), np.floor(cols).astype(int)
    valid = (
        (rows >= 0) & (rows < dem.shape[0]) &
        (cols >= 0) & (cols < dem.shape[1])
    )
    z = np.full(len(points), np.nan)
    z[valid] = dem[rows[valid], cols[valid]]
    return np.nan_to_num(z, nan=0.0)            # assume sea level for NoData

In [10]:
def detailed_visibility_filter(pairs, turb_coords, pop_coords, turb_elevs, pop_elevs, spacing=100, tol=5, batch_size=10000):
    """
    Vectorized coarse pre-filter.
    Returns indices of pairs that pass the filter (possibly visible).
    """
    keep_pairs = []

    # Split into batches to avoid memory issues
    n_batches = int(np.ceil(len(pairs) / batch_size))
    for i in tqdm(range(n_batches), desc="Detailed filter batches"):
        batch = pairs[i*batch_size:(i+1)*batch_size]

        # Prepare arrays
        turb_idx = np.array([t for t, p in batch])
        pop_idx = np.array([p for t, p in batch])

        turb_xy = turb_coords[turb_idx]  # (batch_size, 2)
        pop_xy = pop_coords[pop_idx]     # (batch_size, 2)
        z_turb = turb_elevs[turb_idx] + TURBINE_HEIGHT
        z_pop = pop_elevs[pop_idx] + OBSERVER_HEIGHT

        # Compute line distances and number of samples per line
        dx = pop_xy[:,0] - turb_xy[:,0]
        dy = pop_xy[:,1] - turb_xy[:,1]
        dists_total = np.hypot(dx, dy)
        n_samples = np.maximum(2, (dists_total / spacing).astype(int) + 1)

        # Sample points along each line
        sampled_points = []
        line_fractions = []
        for j in range(len(batch)):
            fracs = np.linspace(0,1,n_samples[j])
            xs = turb_xy[j,0] + dx[j] * fracs
            ys = turb_xy[j,1] + dy[j] * fracs
            sampled_points.append(np.c_[xs, ys])
            line_fractions.append(fracs)
        
        # Flatten for quick KD-tree query
        all_points = np.vstack(sampled_points)
        all_fracs = np.concatenate(line_fractions)
        all_line_lengths = np.repeat(dists_total, [len(f) for f in line_fractions])

        # KD-tree lookup
        z_profile = sample_dem(all_points)

        # Reconstruct per-line
        current_idx = 0
        for j, n_samp in enumerate(n_samples):
            z_line = z_turb[j] + (z_pop[j] - z_turb[j]) * all_fracs[current_idx:current_idx+n_samp]
            # Optional curvature correction
            z_line -= (np.linspace(0, dists_total[j], n_samp)**2) / (2*EARTH_RADIUS)
            # Check if any intermediate point exceeds line + tol
            if not np.any(z_profile[current_idx+1:current_idx+n_samp-1] > z_line[1:-1] + tol):
                keep_pairs.append(batch[j])
            current_idx += n_samp

    return keep_pairs


In [ ]:
# Run only if file "filtered_pairs_detailed.txt" not given - expect to wait several hours
"""
filtered_pairs_detailed = detailed_visibility_filter(
    pairs=pairs,
    turb_coords=turb_coords,
    pop_coords=pop_coords,
    turb_elevs=turb_gdf["mean_elevation"].values,
    pop_elevs=pop_gdf["mean_elevation"].values,
    spacing=100,   # finer sampling
    tol=10,         # stricter
    batch_size=2000
)

print(f"Remaining pairs after detailed filter: {len(filtered_pairs_detailed):,}")

# save as plain text
with open("data/filtered_pairs_detailed_20km.txt", "w") as f:
    for t, p in filtered_pairs_detailed:
        f.write(f"{t},{p}\n")
"""

Detailed filter batches: 100%|██████████| 10540/10540 [1:35:41<00:00,  1.84it/s]


Remaining pairs after detailed filter: 13,725,908


In [12]:
# load it back
filtered_pairs_detailed = [
    tuple(map(int, line.strip().split(",")))
    for line in open("data/filtered_pairs_detailed_20km.txt")
]

In [13]:
# make DataFrame from visible pairs
df = pd.DataFrame(filtered_pairs_detailed, columns=["turb_idx", "pop_idx"])

In [14]:
# map population counts
df["pop_tot"] = pop_gdf.loc[df["pop_idx"], "pop_tot"].values

# sum inhabitants per turbine
seen_counts = df.groupby("turb_idx")["pop_tot"].sum()

# add column to turbine gdf
turb_gdf["visible_pop"] = turb_gdf.index.map(seen_counts).fillna(0).astype(int)

In [15]:
turb_gdf.to_file("data/potential_turbines_visible_population_20km.gpkg", driver="GPKG")